In [1]:
# --- MÓDULO DE INGESTÃO: INDICADOR SOCIAL (MORTALIDADE INFANTIL - DATASUS) ---

import pandas as pd

# Defino o ponto de entrada para os dados de saúde pública.
# Enquanto as variáveis anteriores mensuram o crescimento econômico, esta variável
# é crucial para avaliar o Desenvolvimento Social. O objetivo é testar se a 
# arrecadação da mineração se converte em infraestrutura de saúde e bem-estar.
caminho_arquivo = r'C:\Users\fabio\TCC\10_Mort_Infant.csv'

try:
    # Realizo a ingestão do dataset de mortalidade infantil.
    # skiprows=3: Salto cirúrgico do cabeçalho de metadados padrão do TabNet/DATASUS.
    # encoding='latin-1': Decodificação mandatória para os sistemas legados de saúde.
    df_mort_infantil = pd.read_csv(
        caminho_arquivo,
        sep=';',
        skiprows=3,
        encoding='latin-1'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Valido se a exclusão das 3 primeiras linhas alinhou o cabeçalho tabular corretamente.
    print("--- Visualização do Dataset de Saúde Pública (Mortalidade Infantil) ---")
    print(df_mort_infantil.head())
    
    print("\n--- Inventário de Séries Temporais (DATASUS) ---")
    print(df_mort_infantil.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de Mortalidade Infantil: {e}")

--- Visualização do Dataset de Saúde Pública (Mortalidade Infantil) ---
                      Município 1996 1997 1998 1999 2000 2001 2002 2003 2004  \
0  110001 ALTA FLORESTA D'OESTE   12   14   17   36   10   12   12    7    5   
1              110002 ARIQUEMES   69   54   47   35   30   30   32   34   21   
2                 110003 CABIXI    2    2    2    5    3    3    1    1    4   
3                 110004 CACOAL   29   35   48   35   27   28   37   24   24   
4             110005 CEREJEIRAS   12   10   10    3    5    5    9   12    5   

   ... 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024  
0  ...    7    5    5    3    3    2    2    2    4    3  
1  ...   18   26   23   26   18   11   18   32   19   19  
2  ...    3    -    -    1    2    2    1    -    -    -  
3  ...   25   17   14   19    7   22   13   13   12   11  
4  ...    1    3    1    4    4    2    2    1    -    3  

[5 rows x 30 columns]

--- Inventário de Séries Temporais (DATASUS) ---
['Município', '1996'

In [2]:
# --- ETAPA 2: PARSING DE STRINGS E EXTRAÇÃO DE CHAVES GEOGRÁFICAS (SAÚDE) ---

# Adapto a arquitetura de normalização espacial para os padrões do DATASUS.
# O particionamento correto da string é o que garantirá o alinhamento perfeito
# desta variável social com as variáveis econômicas no painel de Controle Sintético.

print("Iniciando a extração de features geográficas do DATASUS...")

# 1. PARSING E EXPANSÃO VETORIAL (String Splitting)
# O DATASUS tipicamente concatena o Código IBGE e o Nome do Município separados por um espaço.
# n=1: Garante a separação apenas no primeiro espaço, protegendo cidades com nomes compostos.
colunas_separadas = df_mort_infantil['Município'].str.split(' ', n=1, expand=True)

# 2. INSTANCIAÇÃO DAS NOVAS DIMENSÕES ESPACIAIS
# [ALERTA DE INTEGRAÇÃO]: O DATASUS costuma alocar o Código IBGE numérico na posição 0,
# e não a Sigla da UF. Verifique o output para garantir o pareamento correto no Master Merge.
df_mort_infantil['Sigla_UF_ou_Codigo'] = colunas_separadas[0]
df_mort_infantil['Nome_Municipio'] = colunas_separadas[1]

# 3. OTIMIZAÇÃO DE MEMÓRIA (Drop)
# Exclusão da variável aglutinada original para liberar memória RAM e 
# garantir a normalização da forma normal do banco de dados (evitar redundância).
df_mort_infantil = df_mort_infantil.drop(columns=['Município'])

# 4. AUDITORIA DA NORMALIZAÇÃO
print("--- Diagnóstico: Extração de Features Geográficas (Mortalidade Infantil) ---")
print("Validação do desmembramento pelo caracter 'espaço':")
print(df_mort_infantil.head())

Iniciando a extração de features geográficas do DATASUS...
--- Diagnóstico: Extração de Features Geográficas (Mortalidade Infantil) ---
Validação do desmembramento pelo caracter 'espaço':
  1996 1997 1998 1999 2000 2001 2002 2003 2004 2005  ... 2017 2018 2019 2020  \
0   12   14   17   36   10   12   12    7    5   12  ...    5    3    3    2   
1   69   54   47   35   30   30   32   34   21   21  ...   23   26   18   11   
2    2    2    2    5    3    3    1    1    4    -  ...    -    1    2    2   
3   29   35   48   35   27   28   37   24   24   21  ...   14   19    7   22   
4   12   10   10    3    5    5    9   12    5    5  ...    1    4    4    2   

  2021 2022 2023 2024 Sigla_UF_ou_Codigo         Nome_Municipio  
0    2    2    4    3             110001  ALTA FLORESTA D'OESTE  
1   18   32   19   19             110002              ARIQUEMES  
2    1    -    -    -             110003                 CABIXI  
3   13   13   12   11             110004                 CACOAL  
4

In [3]:
# --- ETAPA 3: REESTRUTURAÇÃO PARA PAINEL DE DADOS SOCIAIS (MELT - SAÚDE) ---

# A conversão para o formato vertical (Long/Tidy Data) é mandatória para alinhar
# a variável de mortalidade infantil à arquitetura do Controle Sintético.
# Isso nos permitirá avaliar o impacto social ao longo do tempo.

print("Iniciando a transposição da série histórica do DATASUS...")

# 1. DEFINIÇÃO DAS CHAVES PRIMÁRIAS (ID_VARS)
# [CORREÇÃO PREVENTIVA]: Atualizo a chave para 'Sigla_UF_ou_Codigo' refletindo 
# a extração da etapa anterior e evitando a quebra do pipeline por KeyError.
chaves_geograficas = ['Sigla_UF_ou_Codigo', 'Nome_Municipio']

# 2. OPERAÇÃO DE TRANSPOSIÇÃO (WIDE-TO-LONG)
# Consolido as colunas temporais na dimensão 'Ano' e os indicadores de saúde 
# na variável métrica 'Mortalidade_Infantil'.
df_mort_infantil_final = df_mort_infantil.melt(
    id_vars=chaves_geograficas,
    var_name='Ano',
    value_name='Mortalidade_Infantil'
)

# 3. AUDITORIA DE ESTRUTURA E CRONOLOGIA
# Avalio a integridade do empilhamento inspecionando as bordas temporais do painel.
print("--- Diagnóstico: Painel de Saúde Pública (Long Format) ---")
print("Cabeçalho (Início da Série Histórica):")
print(df_mort_infantil_final.head())

print("\nRodapé (Período Mais Antigo/Fim do Empilhamento):")
print(df_mort_infantil_final.tail())

Iniciando a transposição da série histórica do DATASUS...
--- Diagnóstico: Painel de Saúde Pública (Long Format) ---
Cabeçalho (Início da Série Histórica):
  Sigla_UF_ou_Codigo         Nome_Municipio   Ano Mortalidade_Infantil
0             110001  ALTA FLORESTA D'OESTE  1996                   12
1             110002              ARIQUEMES  1996                   69
2             110003                 CABIXI  1996                    2
3             110004                 CACOAL  1996                   29
4             110005             CEREJEIRAS  1996                   12

Rodapé (Período Mais Antigo/Fim do Empilhamento):
       Sigla_UF_ou_Codigo  Nome_Municipio   Ano Mortalidade_Infantil
161583             522205  VICENTINOPOLIS  2024                    2
161584             522220        VILA BOA  2024                    -
161585             522230   VILA PROPICIO  2024                    1
161586             530010        BRASILIA  2024                  347
161587                

In [4]:
# --- ETAPA 4: ORDENAÇÃO E PERSISTÊNCIA DO PAINEL DE SAÚDE (OUTPUT) ---

# Concluo o processamento do indicador de Desenvolvimento Social (Mortalidade Infantil).
# Esta matriz será o termômetro para testar se a arrecadação se converteu em bem-estar.

# 1. ORDENAÇÃO ESPAÇO-TEMPORAL (Correção de Pipeline)
# [ALERTA METODOLÓGICO]: Inseri o Panel Sorting que não estava no script original. 
# A ordenação transversal (Município) e longitudinal (Ano) é estritamente obrigatória 
# para a regressão do Controle Sintético.
df_mort_infantil_ordenado = df_mort_infantil_final.sort_values(
    by=['Nome_Municipio', 'Ano']
)

# 2. DEFINIÇÃO DO REPOSITÓRIO DE SAÍDA (REFINED ZONE)
# Reintegrando o roteamento para o diretório segregado ('FINALIZADOS'), 
# mantendo a governança de dados intacta.
caminho_saida_csv = r'C:\Users\fabio\TCC\VARIAVEIS\10_Mort_Infantil_FINAL.csv'

# 3. EXPORTAÇÃO E INTEGRIDADE ESTRUTURAL
# Invoco o dataframe ordenado e protegido (df_mort_infantil_ordenado) com index=False 
# para suprimir índices nativos e garantir a portabilidade limpa.
df_mort_infantil_ordenado.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Painel de Indicador Social (Mortalidade Infantil DATASUS) salvo com sucesso em:")
print(caminho_saida_csv)

--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---
Painel de Indicador Social (Mortalidade Infantil DATASUS) salvo com sucesso em:
C:\Users\fabio\TCC\VARIAVEIS\10_Mort_Infantil_FINAL.csv
